### DESCRIPTION (inspired by Leetcode.com)
medium

A regional planning committee wants to identify the most "isolated" city in a road network. Given n cities connected by bidirectional roads with varying distances, find the city that can reach the fewest other cities within a given distance threshold.

If multiple cities have the same minimum count, return the city with the highest number.

Note: The same pair of cities may be directly connected by more than one road. The distance between two cities is the shortest total distance across all possible routes.

Example 1:

Input:
```
n = 4
edges = [[0,1,3], [1,2,1], [1,3,4], [2,3,1]]
distanceThreshold = 4
``
Output: 3

Explanation: After computing shortest paths, city 3 can reach cities 1 (distance 2) and 2 (distance 1) within the threshold — 2 reachable. City 0 can also reach 2 cities, but city 3 wins the tie by having the highest number.

Example 2:

Input:
```
n = 5
edges = [[0,1,2], [0,4,8], [1,2,3], [1,4,2], [2,3,1], [3,4,1]]
distanceThreshold = 2
```

Output:0

Explanation: With threshold 2, city 0 can only reach city 1 (distance 2). Other cities can reach more neighbors. City 0 has the fewest reachable.

In [ ]:
from typing import List

class Solution:
    def findTheCity(self, n: int, edges: List[List[int]], distanceThreshold: int) -> int:
        if not edges or not edges[0] or n <= 1:
            return 0
        
        # 1. Initialize the distance matrix
        dist = [[float('inf')] * n for _ in range(n)]
        for i in range(n):
            dist[i][i] = 0
            
        for src, dst, weight in edges:
            dist[src][dst] = weight
            dist[dst][src] = weight

        # 2. Calculate shortest distance between each pair of cities
        for i in range(n):
            for j in range(n):
                for k in range(n):
                    if k != i and k != j:
                        new_dist = dist[i][k] + dist[k][j]
                        if new_dist < dist[i][j]:
                            dist[i][j] = new_dist
                            dist[j][i] = new_dist

                
        # 3. Find city with fewest connection within threshold
        connections = [0] * n 
        for i in range(n):
            for j in range(n):
                if i != j and dist[i][j] <= distanceThreshold:
                    connections[i] += 1
        
        least = n-1
        for i in range(n):
            if connections[i] < connections[least]:
                least = i
            
        return least

###　Feedback

Your Floyd–Warshall update is not ordered correctly: k must be the outermost loop. A distance discovered using a later intermediate can be missed when earlier i, j pairs have already been processed, causing the failed connected-graph tests. Also, duplicate roads must keep the minimum weight; assigning directly can overwrite a shorter edge with a longer one. Finally, tie-breaking currently keeps the lowest index because you only update on <; use <= (or iterate cities in reverse). This explains the empty-graph failure, where every count ties and city n - 1 should win. Your early return also incorrectly returns 0 for n == 1 only incidentally and for empty edges when n > 1; remove it and let the normal counting logic handle all cases. Complexity remains O(n^3) time and O(n^2) space.


In [39]:
from typing import List

class Solution:
    def findTheCity(self, n: int, edges: List[List[int]], distanceThreshold: int) -> int:
        if not edges or not edges[0]:
            return -1
        
        # 1. Initialize the distance matrix
        dist = [[float('inf')] * n for _ in range(n)]
        for i in range(n):
            dist[i][i] = 0
            
        for u, v, w in edges:
            dist[u][v] = w
            dist[v][u] = w

        # 2. Calculate shortest distance between each pair of cities
        for k in range(n):
            for i in range(n):
                for j in range(n):
                    new_dist = dist[i][k] + dist[k][j]
                    if new_dist < dist[i][j]:
                        dist[i][j] = new_dist
                        
                
        # 3. Find city with fewest connection within threshold
        result_city = 0
        least_count = n
        for city in range(n):
            count = 0
            for i in range(n):
                if city != i and dist[city][i] <= distanceThreshold:
                    count += 1
            if count <= least_count:
                least_count = count
                result_city = city
            
        return result_city

### Feedback

Your Floyd–Warshall and tie-breaking logic are correct, but two edge cases cause the failures:

A graph with no edges is still valid. Every city is isolated, so return the highest-index city. Remove the early return -1.

Parallel roads require keeping the smallest direct weight. Currently a later, longer edge can overwrite a shorter one:

```
for u, v, w in edges:
    dist[u][v] = min(dist[u][v], w)
    dist[v][u] = min(dist[v][u], w)
```

In [5]:
from typing import Callable

class Input:
    def __init__(self, n: int, edges: List[List[int]], distanceThreshold: int):
            self.n = n
            self.edges = edges
            self.distanceThreshold = distanceThreshold
        
class Test:  
    def __init__(self, input: Input, result: int):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[int, List[List[int]], int], int]):
    for test in tests:
        result = func(test.input.n, test.input.edges, test.input.distanceThreshold)
        if result == test.expected_result:
            print(f"Test passed for {test.input.edges}")
        else:
            print(f"Test failed for {test.input.edges}. Expected: {test.expected_result}, Actual: {result}")

In [40]:
tests = [
    Test(Input(2, [[0,1,10]], 1), 1),
    Test(Input(2, [[0,1,5]], 5), 1),
    Test(Input(4, [[0,1,3], [1,2,1], [1,3,4], [2,3,1]], 4), 3),
    Test(Input(5, [[0,1,2], [0,4,8], [1,2,3], [1,4,2], [2,3,1], [3,4,1]], 2), 0),
    Test(Input(4, [[0,1,1],[0,2,1],[2,3,1]], 2), 3),
    Test(Input(4, [[0,3,1],[1,3,1],[1,2,1]], 2), 2),
]

run_tests(tests, Solution().findTheCity)

Test passed for [[0, 1, 10]]
Test passed for [[0, 1, 5]]
Test passed for [[0, 1, 3], [1, 2, 1], [1, 3, 4], [2, 3, 1]]
Test passed for [[0, 1, 2], [0, 4, 8], [1, 2, 3], [1, 4, 2], [2, 3, 1], [3, 4, 1]]
Test passed for [[0, 1, 1], [0, 2, 1], [2, 3, 1]]
Test passed for [[0, 3, 1], [1, 3, 1], [1, 2, 1]]
